# 面向智能体系统的宏观评估

当一个智能体系统出现故障时，问题往往不止于某一次糟糕的回答。交接可能发生得太晚，某个专家智能体可能在许多次运行中都错过同一个信号，或者审查流程可能针对错误类别的案例被触发。要改进系统，团队需要在整个追踪（trace）总体中看到反复出现的行为。

本 cookbook 将带你走完一个多智能体系统的宏观评估（macro-eval）工作流。我们使用一个合成的电动汽车（EV）订单工作流，其中各专家智能体在市场和运营条件不断变化的情况下，负责定价、合规、供应、工厂路由、排产和放行决策。

本 notebook 使用预先计算好的合成追踪和已保存的底层评估标签，因此你无需 OpenAI API 密钥就能运行完整的工作流。

你将学习如何：
1. 生成或收集大量带追踪的智能体运行；
2. 对每次完成的运行执行底层评估；
3. 将每条追踪转换为一份紧凑的文档；
4. 在总体中发现反复出现的行为模式；以及
5. 深入分析一个高影响模式，找出接下来应由人工检查系统的哪个部分。

我们的目标不是为每条追踪构建一套完美的分类体系，而是展示 AI 工程团队如何从成千上万个智能体事件中提炼出少量技术和业务利益相关者都能理解的模式。


## 端到端智能体系统示意图

![端到端宏观评估架构](../../../images/agentic-system-architecture.svg)

关键在于，本 notebook 评估的是一个已保存的智能体系统，而不是一份通用的聊天记录。场景输入驱动一个经过编排的专家智能体群，运行时输出追踪包，已保存的 Promptfoo 标签被关联到规范化后的追踪上，而宏观评估层则把这些证据转化为模式视图和诊断视图。


## 1. 为什么需要宏观评估？

评估是 AI 团队衡量系统是否正常工作的方式。对于一次简单的模型调用，评估可能只是将一个输出与评分标准或参考答案进行比较。而对于智能体系统，我们还需要评估系统是否使用了正确的工具、是否委派给了正确的专家、是否在高风险时暂停以待审查，以及是否始终立足于业务上下文。

多智能体系统让这件事变得更难，因为最终答案只是一个更长工作流中的最后一个事件。一条放行建议看起来可能合情合理，但追踪却可能显示：定价智能体忽略了某项激励政策，供应智能体错过了某次缺货，或者编排器绕过了一个必需的审查步骤。

本 notebook 将问题分为两个层级：

- **底层评估**对单个智能体、交接、工具和已完成的运行进行评分。在本例中，Promptfoo 充当这一智能体级评估层，评判一次运行在最终决策质量、策略正确性、专家路由、市场漂移和审查恰当性方面的表现。
- **宏观评估**则纵观大量底层评估结果。它要回答的是：哪些类型的问题会反复出现，它们集中在哪里，以及我们应该首先检查智能体工作流的哪个部分？

在整个 cookbook 中，我们将使用四个面向读者的标签：

- `case_type`：生成的业务情境，例如正常订单、校验阻断、供应商替换或定价例外。
- `run_outcome`：运行的结束方式，例如已完成、等待审查、被阻断或失败。
- `eval_finding`：底层信号，说明哪里看起来有问题或存在风险。
- `behavior_pattern`：在大量追踪中发现的反复出现的模式。

一个有用的思维模型是：`case_type` 是开局设定，`run_outcome` 是结局，`eval_finding` 是局部症状，`behavior_pattern` 是总体层面的模式。


In [ ]:
import sys
from pathlib import Path

if sys.version_info < (3, 11):
    raise RuntimeError("This notebook requires Python 3.11 or newer.")

if not Path("requirements.txt").is_file():
    raise FileNotFoundError("requirements.txt must be in the same folder as this notebook.")

%pip install -q --upgrade pip setuptools wheel
%pip install -q --only-binary=:all: -r requirements.txt


## 准备工作与数据材料

安装依赖项，然后加载本示例附带的离线数据集。已保存的 Promptfoo 标签包含在本地数据文件夹中，因此本 notebook 不需要单独的 Promptfoo 配置、Promptfoo 运行产物或 OpenAI API 密钥。

预期的文件：

```text
data/trace_results.jsonl
data/run_summary.json
data/trace_bundles.zip
data/eval_labels.jsonl
```

`trace_bundles.zip` 会在 notebook 首次运行时自动解压到本地缓存中。你可以将完整的 SQLite 追踪快照放在 `data/trace_snapshot.sqlite` 以进行可选的数据增强，但端到端工作流并不需要它。

如果你的数据位于示例文件夹之外，请将 `MACRO_EVALS_DATA_ROOT` 设置为该目录。如果标签单独存放，请设置 `MACRO_EVALS_LABELS_PATH`。


In [ ]:
from __future__ import annotations

import json
import os
import sqlite3
import sys
import warnings
import zipfile
from pathlib import Path
from time import perf_counter
from typing import Any

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_rows", 100)
warnings.filterwarnings("ignore", message="n_jobs value 1 overridden.*")


def find_example_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents, start / "examples/partners/macro_evals_for_agentic_systems"]
    for candidate in candidates:
        if (candidate / "helpers/data_prep.py").is_file() and (candidate / "helpers/macro_eval_pipeline.py").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the macro evals example root.")


EXAMPLE_ROOT = find_example_root()
HELPERS_ROOT = EXAMPLE_ROOT / "helpers"
if str(HELPERS_ROOT) not in sys.path:
    sys.path.insert(0, str(HELPERS_ROOT))

from data_prep import add_public_label_columns, build_trace_documents, load_promptfoo_label_rows, normalize_bundle
from macro_eval_pipeline import (
    drill_down_topic_root_causes,
    pick_focus_topic,
    plot_root_cause_story,
    plot_suspect_leaderboard,
    plot_topic_heatmap,
    plot_topic_leaderboard,
    plot_topic_scatter,
    plot_trace_swimlane,
    run_macro_discovery,
    slice_topics_by_metadata,
)


def display_path(path: Path | None) -> str:
    if path is None:
        return "not found"
    try:
        return str(path.resolve().relative_to(EXAMPLE_ROOT))
    except ValueError:
        return str(path)


def as_path(value: str | Path) -> Path:
    path = Path(value).expanduser()
    return path if path.is_absolute() else EXAMPLE_ROOT / path


def unique_paths(paths: list[Path]) -> list[Path]:
    seen: set[Path] = set()
    unique: list[Path] = []
    for path in paths:
        resolved = path.resolve()
        if resolved not in seen:
            seen.add(resolved)
            unique.append(resolved)
    return unique


def find_material(label: str, names: list[str], *, kind: str = "file", required: bool = True) -> Path | None:
    checked: list[Path] = []
    for root in DATA_ROOTS:
        for name in names:
            if not name:
                continue
            candidate = as_path(name) if Path(name).expanduser().is_absolute() else root / name
            checked.append(candidate)
            if kind == "dir":
                exists = candidate.is_dir() and any(candidate.glob("*.json"))
            else:
                exists = candidate.is_file()
            if exists:
                return candidate.resolve()
    if required:
        checked_text = "\n".join(f"- {display_path(path)}" for path in checked)
        raise FileNotFoundError(f"Missing {label}. Checked:\n{checked_text}")
    return None


def ensure_trace_bundle_dir(bundle_dir: Path | None, bundle_zip: Path | None) -> Path:
    if bundle_dir is not None:
        return bundle_dir
    if bundle_zip is None:
        raise FileNotFoundError("Missing trace bundles. Expected data/trace_bundles/ or data/trace_bundles.zip.")
    cache_dir = bundle_zip.parent / ".macro_eval_cache" / "trace_bundles"
    marker = cache_dir / ".extracted_from_trace_bundles_zip"
    if not marker.is_file() or not any(cache_dir.glob("*.json")):
        cache_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(bundle_zip) as archive:
            for member in archive.infolist():
                if member.is_dir() or not member.filename.endswith(".json"):
                    continue
                (cache_dir / Path(member.filename).name).write_bytes(archive.read(member))
        marker.write_text(str(bundle_zip.stat().st_mtime_ns), encoding="utf-8")
    return cache_dir.resolve()


env_data_root = os.environ.get("MACRO_EVALS_DATA_ROOT")
DATA_ROOTS = unique_paths(
    ([as_path(env_data_root)] if env_data_root else [])
    + [
        EXAMPLE_ROOT / "data",
    ]
)

RESULTS_PATH = find_material("trace results", ["trace_results.jsonl", "metadata/results.jsonl", "results.jsonl"])
SUMMARY_PATH = find_material("run summary", ["run_summary.json", "metadata/summary.json", "summary.json"])
SQLITE_PATH = find_material("optional trace snapshot", ["trace_snapshot.sqlite"], required=False)
BUNDLE_ZIP_PATH = find_material("trace bundle archive", ["trace_bundles.zip", "bundles.zip"], required=False)
BUNDLE_DIR = ensure_trace_bundle_dir(find_material("trace bundles", ["trace_bundles", "bundles"], kind="dir", required=False), BUNDLE_ZIP_PATH)
PROGRESS_PATH = find_material("run progress", ["run_progress.json", "metadata/progress.json", "progress.json"], required=False)
PROMPTFOO_LABELS_PATH = find_material(
    "lower-level eval labels",
    [
        os.environ.get("MACRO_EVALS_LABELS_PATH", ""),
        "eval_labels.jsonl",
        "metadata/eval_labels.jsonl",
    ],
    required=False,
)

DATA_ROOT = next((root for root in DATA_ROOTS if RESULTS_PATH.is_relative_to(root)), DATA_ROOTS[0])
TRACE_LIMIT = int(os.environ.get("MACRO_EVALS_TRACE_LIMIT", "0")) or None
DISCOVERY_DOC_COLUMN = "doc_structured_summary"
DISCOVERY_MIN_CLUSTER_SIZE = int(os.environ.get("MACRO_EVALS_DISCOVERY_MIN_CLUSTER_SIZE", "24"))
RANDOM_STATE = 42

resolved_paths_df = pd.DataFrame(
    [
        ("Trace results", RESULTS_PATH),
        ("Run summary", SUMMARY_PATH),
        ("Trace bundle archive", BUNDLE_ZIP_PATH),
        ("Expanded trace bundles", BUNDLE_DIR),
        ("Optional trace snapshot", SQLITE_PATH),
        ("Run progress", PROGRESS_PATH),
        ("Lower-level eval labels", PROMPTFOO_LABELS_PATH),
    ],
    columns=["material", "path"],
)
resolved_paths_df["path"] = resolved_paths_df["path"].map(display_path)

display(Markdown("### Data materials"))
display(resolved_paths_df)
display(Markdown(f"Example root: `{display_path(EXAMPLE_ROOT)}`  \nData root: `{display_path(DATA_ROOT)}`"))


## 2. 仿真场景：瞬息万变环境中的汽车订单

被仿真的业务是一个电动汽车订单及配置后工作流。客户已经选定了车辆配置，公司需要决定该订单是可以按原样推进、需要调整、应当改道、需要替换零部件，还是应暂停以待审查。

该仿真包含了让真实汽车订单履约变得困难的各类约束：

- 零部件可用性与供应商替换；
- 工厂产能与生产排程；
- 定价例外、促销和激励政策；
- 关税和带有时间戳的市场信号；
- 区域合规约束；
- 客户澄清和升级路径；
- 针对高风险或模糊案例的放行审查阈值。

智能体群围绕这些业务职责进行组织。编排器接收订单和当前环境，然后委派给各个专家，例如校验、供应风险、采购计划、产能平衡、工厂路由、市场情报、定价、合规、客户沟通和放行审查。

这与 OpenAI Agents SDK 天然契合。在 SDK 中，智能体是工作流的核心单元：它封装了一个模型、指令以及可选的运行时行为，例如工具、交接、护栏和结构化输出。本仿真遵循这一模式：

- **专家智能体**封装决策中某一部分所需的指令和工具；
- **交接（handoffs）**让编排器能够委派给另一个专家智能体，而不是把所有职责都塞进一个提示词；
- **函数工具**通过结构化的输入和输出，暴露订单数据、环境信号和审批标记；
- **护栏和审查阈值**代表针对高风险或模糊案例的校验、阻断和人工审查流程；
- **结构化输出**使下游的评分和聚合成为可能；
- **追踪**保留了模型调用、工具调用、交接、护栏和自定义 span 的结构化记录，用于调试和宏观层面的分析。

本 notebook 后面的底层评估都立足于这一仿真故事。如果 case type 表明存在关税压力下的供应商替换，那么追踪就应当体现出对供应、策略、市场和审查风险的认知。如果 case type 是正常订单，那么不必要的升级本身就是一项发现。


In [ ]:
def read_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def result_run_id(row: dict[str, Any]) -> str | None:
    if row.get("run_id"):
        return str(row["run_id"])
    if row.get("bundle_path"):
        return Path(str(row["bundle_path"])).stem
    return None


def sqlite_table_counts(db_path: Path | None) -> pd.DataFrame:
    tables = [
        "runs",
        "configs",
        "traces",
        "trace_events",
        "spans",
        "review_packets",
        "environment_events",
        "environment_decisions",
    ]
    if db_path is None:
        return pd.DataFrame([{"table": table, "row_count": 0} for table in tables])
    with sqlite3.connect(db_path) as conn:
        existing_tables = {
            row[0]
            for row in conn.execute("select name from sqlite_master where type = 'table'")
        }
        rows = [
            {
                "table": table,
                "row_count": conn.execute(f"select count(*) from {table}").fetchone()[0] if table in existing_tables else 0,
            }
            for table in tables
        ]
    return pd.DataFrame(rows)


def load_sqlite_runs(db_path: Path | None) -> pd.DataFrame:
    if db_path is None:
        return pd.DataFrame()
    summary_fields = [
        "scenario_family",
        "validation_outcome",
        "review_status",
        "review_decision",
        "triage_outcome",
        "market_regime",
        "price_regime",
        "schedule_regime",
        "agent_version_set",
        "orchestrator_mode",
        "rogue_window_id",
        "factory_release_state",
        "trace_family",
        "loop_count",
        "retry_count",
        "arbitration_count",
        "compound_issue_count",
        "specialist_activations",
        "environment_event_ids",
        "findings",
        "failure_agent",
        "error_code",
        "error_message",
    ]
    rows = []
    with sqlite3.connect(db_path) as conn:
        for row in conn.execute("select run_id, config_id, trace_id, status, terminal_state, started_at, ended_at, summary_json from runs"):
            run_id, config_id, trace_id, status, terminal_state, started_at, ended_at, summary_json = row
            summary = json.loads(summary_json or "{}")
            item = {field: summary.get(field) for field in summary_fields}
            item.update(
                {
                    "run_id": run_id,
                    "config_id": config_id,
                    "trace_id": trace_id,
                    "sqlite_status": status,
                    "sqlite_terminal_state": terminal_state,
                    "started_at": started_at,
                    "ended_at": ended_at,
                }
            )
            rows.append(item)
    runs = pd.DataFrame(rows)
    if runs.empty:
        return runs
    runs["started_at"] = pd.to_datetime(runs["started_at"], utc=True, errors="coerce")
    runs["ended_at"] = pd.to_datetime(runs["ended_at"], utc=True, errors="coerce")
    runs["findings_count_sqlite"] = runs["findings"].apply(lambda value: len(value or []))
    runs["specialist_activation_count_sqlite"] = runs["specialist_activations"].apply(lambda value: len(value or []))
    runs["environment_event_count_sqlite"] = runs["environment_event_ids"].apply(lambda value: len(value or []))
    return runs


batch_summary = read_json(SUMMARY_PATH)
results_rows = read_jsonl(RESULTS_PATH)
sqlite_runs_df = load_sqlite_runs(SQLITE_PATH)
table_counts_df = sqlite_table_counts(SQLITE_PATH)

result_ids = {rid for row in results_rows if (rid := result_run_id(row))}
bundle_ids = {path.stem for path in BUNDLE_DIR.glob("*.json")}
missing_result_rows = [row for row in results_rows if result_run_id(row) is None]

def table_count(table_name: str) -> int:
    rows = table_counts_df.loc[table_counts_df["table"].eq(table_name), "row_count"]
    return int(rows.iloc[0]) if not rows.empty else 0


dataset_profile_df = pd.DataFrame(
    [
        ("requested_batch_size", batch_summary.get("batch_size") or batch_summary.get("requested_runs")),
        ("results_rows", len(results_rows)),
        ("bundle_backed_result_rows", len(result_ids)),
        ("runner_error_rows_without_bundle", len(missing_result_rows)),
        ("bundle_files_available", len(bundle_ids)),
        ("bundle_files_not_in_results", len(bundle_ids - result_ids)),
        ("saved_promptfoo_label_rows", len(read_jsonl(PROMPTFOO_LABELS_PATH)) if PROMPTFOO_LABELS_PATH else 0),
        ("sqlite_available", SQLITE_PATH is not None),
        ("sqlite_runs", len(sqlite_runs_df)),
        ("sqlite_trace_events", table_count("trace_events")),
        ("sqlite_spans", table_count("spans")),
    ],
    columns=["metric", "value"],
)

display(dataset_profile_df)
if SQLITE_PATH is not None:
    display(table_counts_df)

if missing_result_rows:
    display(Markdown(
        f"The batch has `{len(results_rows):,}` result rows, but `{len(missing_result_rows):,}` ended before a bundle was written. "
        f"The macro analysis therefore focuses on the `{len(result_ids):,}` bundle-backed traces that can be normalized and graded retrospectively."
    ))

if SQLITE_PATH is None:
    display(Markdown(
        "This packaged version omits the large SQLite mirror. The notebook uses the JSONL result rows, trace bundles, and saved Promptfoo labels for the end-to-end workflow."
    ))


### 一个 Bundle 代表什么

在本 notebook 中，一个 **bundle** 是一次仿真客户订单交互的证据包。

设想一位客户配置了一辆电动汽车，业务方需要决定下一步该怎么做。智能体群接收该订单以及当前的运营环境：供应约束、工厂产能、促销、激励政策、关税、竞争压力和审查阈值。随后，各智能体将工作路由给各个专家，并产出最终状态。bundle 就是我们事后审计这次交互所需的全部内容。

bundle 之所以重要，是因为宏观评估需要最终答案背后的工作流证据。它们需要知道咨询了哪些智能体、调用了哪些工具、哪些环境信号处于激活状态、是否需要审查，以及工作流在何处改变了方向。有了这些证据，我们就可以从“这一次运行发生了什么？”转向“哪些工作流模式会在许多次运行中反复出现？”


In [ ]:
def bundle_path_for_result(result_row: dict[str, Any]) -> Path | None:
    raw = result_row.get("bundle_path")
    if not raw:
        return None
    return BUNDLE_DIR / Path(str(raw)).name


def bundle_event_counts(bundle: dict[str, Any]) -> dict[str, int]:
    events = bundle.get("events") or []
    spans = bundle.get("spans") or []
    event_types = pd.Series([event.get("event_type") or "unknown" for event in events])
    span_types = pd.Series([span.get("span_type") or "unknown" for span in spans])
    agents = {
        event.get("agent_name")
        for event in events
        if event.get("agent_name")
    } | {
        span.get("agent_name")
        for span in spans
        if span.get("agent_name")
    }
    return {
        "events": len(events),
        "spans": len(spans),
        "handoffs": int(event_types.eq("handoff").sum() + span_types.eq("handoff").sum()),
        "tool_or_function_calls": int(event_types.eq("function").sum() + span_types.eq("function").sum()),
        "status_updates": int(event_types.eq("status").sum()),
        "unique_agents_seen": len(agents),
        "environment_signals": len(bundle.get("environment_events") or []),
        "has_review_packet": int(bool(bundle.get("review_packet"))),
    }


def human_event_type(value: str) -> str:
    cleaned = str(value or "").replace("product_launch", "product_launch")
    return cleaned.replace("_", " ")


bundle_rows = []
sample_bundle = None
sample_result_row = None
for result_row in results_rows:
    bundle_path = bundle_path_for_result(result_row)
    if bundle_path is None or not bundle_path.is_file():
        continue
    bundle = read_json(bundle_path)
    counts = bundle_event_counts(bundle)
    counts["run_id"] = result_run_id(result_row)
    counts["case_type"] = result_row.get("scenario_family")
    counts["final_status"] = result_row.get("final_status") or result_row.get("status")
    counts["review_status"] = result_row.get("review_status")
    bundle_rows.append(counts)
    if sample_bundle is None and result_row.get("scenario_family") != "clean_simple":
        sample_bundle = bundle
        sample_result_row = result_row

bundle_profile_df = pd.DataFrame(bundle_rows)
typical_bundle_df = pd.DataFrame(
    [
        ("analyzable customer-order interactions", len(bundle_profile_df), "One completed trace bundle per simulated order interaction."),
        ("median normalized events per interaction", int(bundle_profile_df["events"].median()), "Status updates, handoffs, function/tool events, responses, and findings."),
        ("median SDK spans per interaction", int(bundle_profile_df["spans"].median()), "Lower-level SDK trace spans behind the event log."),
        ("median handoff records per interaction", int(bundle_profile_df["handoffs"].median()), "Delegations between orchestrator and specialist agents."),
        ("median tool/function calls per interaction", int(bundle_profile_df["tool_or_function_calls"].median()), "Structured reads, checks, and evaluation calls inside the run."),
        ("median agents observed per interaction", int(bundle_profile_df["unique_agents_seen"].median()), "How many specialist roles appear in a typical trace."),
        ("median environment signals per interaction", int(bundle_profile_df["environment_signals"].median()), "Tariff, incentive, stockout, promotion, competitor, launch, or schedule signals active for the order."),
        ("interactions with review packets", int(bundle_profile_df["has_review_packet"].sum()), "Runs where the simulated business process produced a review artifact."),
    ],
    columns=["reader_metric", "value", "plain_english_meaning"],
)
display(typical_bundle_df)

bundle_anatomy_df = pd.DataFrame(
    [
        ("run", "Run id, trace id, terminal state, batch metadata, and synthetic order context.", "Lets us join one interaction across tables and understand its business setup."),
        ("events", "A normalized event log: status updates, handoffs, tool/function activity, responses, and findings.", "This is the main evidence stream used for trace documents and AgentTrace-style diagnosis."),
        ("spans", "OpenAI Agents SDK trace spans for handoffs, function calls, responses, and timing.", "Gives lower-level execution structure behind the event log."),
        ("environment_events", "The dated world state active for the order: tariffs, incentives, stockouts, promotions, competitor pressure, launches, and schedule/capacity signals.", "Lets evals check whether the swarm reacted to the world it was given."),
        ("review_packet", "A simulated review artifact with findings, recommended action, allowed actions, and review status.", "Lets us evaluate whether escalation or review was appropriate."),
        ("snapshots", "Optional inventory, capacity, and environment snapshots.", "Provides operational context when a case depends on supply or scheduling."),
    ],
    columns=["bundle_part", "what_it_contains", "why_it_matters_for_macro_evals"],
)
display(bundle_anatomy_df)


In [ ]:
if sample_bundle is not None and sample_result_row is not None:
    run_config = (sample_bundle.get("run") or {}).get("config") or {}
    metadata = run_config.get("metadata") or {}
    generation = metadata.get("generation_params") or {}
    customer = run_config.get("customer") or {}
    active_event_types = sorted({human_event_type(value) for value in generation.get("active_event_types") or []})
    specialists = sample_result_row.get("specialist_activations") or generation.get("specialist_activations") or []
    review_packet = sample_bundle.get("review_packet") or {}

    example_interaction_df = pd.DataFrame(
        [
            ("what the interaction represents", "One synthetic customer order moving through the post-configuration workflow."),
            ("case_type", sample_result_row.get("scenario_family")),
            ("synthetic customer region", customer.get("region") or "not recorded"),
            ("business issue cluster", generation.get("issue_cluster") or sample_result_row.get("scenario")),
            ("active world signals", ", ".join(active_event_types[:8]) + (" ..." if len(active_event_types) > 8 else "")),
            ("specialists activated", ", ".join(map(str, specialists[:8])) + (" ..." if len(specialists) > 8 else "")),
            ("final status", sample_result_row.get("final_status") or sample_result_row.get("status")),
            ("review status", sample_result_row.get("review_status") or review_packet.get("status") or "not recorded"),
            ("event evidence", f"{len(sample_bundle.get('events') or []):,} events and {len(sample_bundle.get('spans') or []):,} SDK spans"),
        ],
        columns=["field", "example_value"],
    )
    display(example_interaction_df)


### 如何解读数据集概况

数据集概况告诉我们即将评估的仿真业务流程的规模和特征。每一条可分析的行都是一次客户订单交互，并带有足够的追踪证据，能够还原智能体群看到了什么、咨询了哪些专家，以及工作流是如何结束的。

生成的批次要求智能体群处理 1,000 次合成订单交互。其中 992 次拥有 bundle：一个完整的证据包，可用于对运行进行评分、构建追踪文档、将其与相似运行聚类，以及事后检查智能体的执行路径。这为我们提供了足够大的总体来寻找重复行为，同时仍保留了解释单个示例所需的追踪细节。

典型的 bundle 是一份仿真业务流程的结构化记录：订单设定、活跃的外部事件、专家交接、工具/函数活动、审查产物以及终止状态。这正是该数据集能够支撑宏观评估的原因。我们既可以评估单个决策，也可以追问在数百条丰富的交互记录中是否浮现出重复的工作流模式。


In [ ]:
scenario_counts_df = (
    pd.DataFrame(results_rows)
    .assign(case_type=lambda df: df["scenario_family"].fillna("unknown"))
    .groupby("case_type", as_index=False)
    .size()
    .rename(columns={"size": "run_count"})
    .sort_values("run_count", ascending=False)
)

fig = px.bar(
    scenario_counts_df,
    x="run_count",
    y="case_type",
    orientation="h",
    title="Synthetic simulation coverage by generated case type",
    text="run_count",
    color="run_count",
    color_continuous_scale="Teal",
)
fig.update_layout(height=max(420, 28 * len(scenario_counts_df)), margin=dict(l=20, r=20, t=60, b=30))
fig.update_yaxes(title="", categoryorder="total ascending")
fig.update_xaxes(title="Run count")
fig.show()

display(scenario_counts_df.head(15))


### `case_type` 的含义

`case_type` 是来自生成器的场景标签。它描述了在任何评估或聚类发生之前，智能体群被要求处理的业务情境类型。

该数据集中的示例包括：

- `clean_simple`：相对简单的订单，正确行为通常是在不进行不必要审查的情况下完成。
- `validation_block_simple`：配置存在校验问题，因此智能体群应避免过度自信地放行。
- `supplier_substitution_compound`：零部件可用性引发替换决策，通常会对下游的路由和排产产生影响。
- `pricing_exception_compound`：定价、激励或利润率政策需要专家审查。
- `regional_compliance_compound`：订单需要区域策略或合规处理。

上面的柱状图是一个覆盖度视图。它显示仿真是否产生了足够多样的情况，以便在不同的业务压力下评估智能体群。一个优秀的宏观评估数据集既需要普通案例，也需要压力案例，因为只有当我们能够在不同设定之间比较行为时，反复出现的模式才有意义。


In [ ]:
CASE_TYPE_DESCRIPTIONS = {
    "clean_simple": "Straightforward order; should usually complete with minimal routing.",
    "validation_block_simple": "Configuration or buildability issue; should avoid unsupported release.",
    "release_block_simple": "Release readiness is blocked; should defer or request review.",
    "capacity_hold_simple": "Factory capacity or scheduling pressure; should route to fulfillment planning.",
    "pricing_exception_compound": "Pricing, incentives, margin, or tariff pressure; should involve pricing/policy owners.",
    "escalation_resume_compound": "An escalation or review flow needs to resume cleanly.",
    "supplier_substitution_compound": "Supplier availability forces substitution or procurement planning.",
    "regional_compliance_compound": "Regional compliance or policy constraints affect release.",
    "clarification_needed_compound": "Customer intent or configuration details are ambiguous.",
    "schedule_incentive_compound": "Timing, scheduling, and incentive windows interact.",
    "tradeoff_recommended_compound": "The system should weigh competing business tradeoffs.",
    "dual_failure_recovery_compound": "Multiple failures require coordinated recovery.",
    "ambiguous_customer_intent_compound": "The customer request is underspecified or conflicting.",
    "conflicting_multi_agent_compound": "Specialists may surface conflicting recommendations.",
}

case_type_guide_df = scenario_counts_df.copy()
case_type_guide_df["plain_english"] = case_type_guide_df["case_type"].map(CASE_TYPE_DESCRIPTIONS).fillna(
    "Generated scenario type from the synthetic simulation."
)
display(case_type_guide_df.head(12))


上表将生成器的标签转换成了业务语言。这一点很重要，因为同一个后续模式在不同设定下可能意味着不同的事情。在供应商替换案例中，履约改道可能是可取的；而在正常案例中，同样的改道可能只是不必要的复杂性。


## 3. 使用 Promptfoo 进行底层智能体评估

成熟的多智能体系统不应仅仅依赖对最终答案的检查。每个上线的智能体通常都需要自己的评估：这个专家是否使用了正确的证据、调用了正确的工具、遵守了策略、在正确的时机进行交接，并产出了系统其余部分可以信赖的输出？

在本 notebook 中，Promptfoo 扮演的就是这个角色。它代表了在生产工作流中通常与智能体并存的底层评估层。在实际系统中，其中一些检查可能在线运行，一些可能异步运行，还有一些可能被抽样交由人工审查。实现细节远不如约定重要：每次运行都应带有评估信号，说明在智能体和工作流层面哪些看起来是正确的、有风险的或错误的。

在该数据集中，Promptfoo 使用的问题与团队为真实系统构建的智能体级评估类似，对已完成的追踪进行评分：

- 最终决策是否源自当前的实际问题？
- 系统是否遵守了定价、关税、激励、区域和策略方面的约束？
- 编排器是否激活了该案例所需的专家？
- 运行是否对带有时间戳的市场信号做出了响应，而不是仿佛世界一成不变？
- 审查或升级是否与风险相称？

这些检查产生了 `eval_finding`。一次失败的底层评估是一个局部信号：一条追踪、一项评分标准、一个症状。后面的宏观评估部分会追问这些局部信号在总体规模上会变成什么。它们是随机分散的，还是揭示了指向某个特定智能体、交接、工具或业务策略的重复行为模式？


In [ ]:
PROMPTFOO_RUBRICS = [
    ("final_decision_quality", "Final decision is supported by the active issues, terminal state, and agent outputs."),
    ("policy_compliance_correctness", "Policy, tariff, incentive, and regional compliance context is handled correctly."),
    ("routing_specialist_activation", "Specialist routing matches the issues present in the bundle."),
    ("market_drift_awareness", "Changing market conditions and dated environment signals are noticed."),
    ("review_appropriateness", "Review and escalation behavior is proportionate to the case risk."),
]
PROMPTFOO_ASSERTION_FALLBACK = {
    f"assertion_{idx}": metric for idx, (metric, _) in enumerate(PROMPTFOO_RUBRICS, start=1)
}


def clean_metric_name(value: Any) -> str | None:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    text = str(value)
    return PROMPTFOO_ASSERTION_FALLBACK.get(text, text)


def clean_metric_list(values: Any) -> list[str]:
    if not isinstance(values, list):
        return []
    return [metric for item in values if (metric := clean_metric_name(item))]


def clean_metric_dict(values: Any) -> dict[str, Any]:
    if not isinstance(values, dict):
        return {}
    return {clean_metric_name(key) or str(key): value for key, value in values.items()}


def clean_promptfoo_labels(labels_df: pd.DataFrame) -> pd.DataFrame:
    if labels_df.empty:
        return labels_df
    cleaned = labels_df.copy()

    # Accept older draft label files that used metric/passed/score/reason columns.
    if "promptfoo_pass" not in cleaned.columns and "passed" in cleaned.columns:
        cleaned["promptfoo_pass"] = cleaned["passed"]
    if "promptfoo_failed_checks" not in cleaned.columns:
        if "metric" in cleaned.columns:
            cleaned["promptfoo_failed_checks"] = cleaned.apply(
                lambda row: [] if bool(row.get("promptfoo_pass", True)) else [row.get("metric")],
                axis=1,
            )
        else:
            cleaned["promptfoo_failed_checks"] = [[] for _ in range(len(cleaned))]
    if "promptfoo_score_mean" not in cleaned.columns and "score" in cleaned.columns:
        cleaned["promptfoo_score_mean"] = cleaned["score"]
    if "promptfoo_primary_finding" not in cleaned.columns:
        if "metric" in cleaned.columns:
            cleaned["promptfoo_primary_finding"] = cleaned.apply(
                lambda row: None if bool(row.get("promptfoo_pass", True)) else row.get("metric"),
                axis=1,
            )
        else:
            cleaned["promptfoo_primary_finding"] = None
    if "promptfoo_check_scores" not in cleaned.columns:
        cleaned["promptfoo_check_scores"] = cleaned.apply(
            lambda row: {row.get("metric", "unknown"): row.get("score")} if "score" in cleaned.columns else {},
            axis=1,
        )
    if "promptfoo_rationales" not in cleaned.columns:
        cleaned["promptfoo_rationales"] = cleaned.apply(
            lambda row: {row.get("metric", "unknown"): row.get("reason")} if "reason" in cleaned.columns else {},
            axis=1,
        )

    cleaned["promptfoo_pass"] = cleaned["promptfoo_pass"].astype("boolean")
    cleaned["promptfoo_primary_finding"] = cleaned["promptfoo_primary_finding"].apply(clean_metric_name)
    cleaned["promptfoo_failed_checks"] = cleaned["promptfoo_failed_checks"].apply(clean_metric_list)
    cleaned["promptfoo_check_scores"] = cleaned["promptfoo_check_scores"].apply(clean_metric_dict)
    cleaned["promptfoo_rationales"] = cleaned["promptfoo_rationales"].apply(clean_metric_dict)
    return cleaned


rubric_df = pd.DataFrame(PROMPTFOO_RUBRICS, columns=["rubric", "plain_english_question"])
display(rubric_df)

promptfoo_labels_df = clean_promptfoo_labels(load_promptfoo_label_rows(PROMPTFOO_LABELS_PATH))

if promptfoo_labels_df.empty:
    display(Markdown("No Promptfoo labels were found. The notebook will continue with deterministic review and runtime signals only."))
else:
    pass_counts_df = (
        promptfoo_labels_df["promptfoo_pass"]
        .map({True: "pass", False: "fail"})
        .fillna("unknown")
        .value_counts()
        .rename_axis("promptfoo_result")
        .reset_index(name="trace_count")
    )
    display(Markdown(f"Loaded `{len(promptfoo_labels_df):,}` Promptfoo label rows from `{display_path(PROMPTFOO_LABELS_PATH)}`."))
    display(pass_counts_df)

    fig = px.pie(
        pass_counts_df,
        names="promptfoo_result",
        values="trace_count",
        title="Promptfoo grading result across bundle-backed traces",
        color="promptfoo_result",
        color_discrete_map={"pass": "#4daf4a", "fail": "#e41a1c", "unknown": "#999999"},
        hole=0.45,
    )
    fig.update_traces(textposition="inside", textinfo="percent+label")
    fig.show()

    failed_metric_df = (
        promptfoo_labels_df.explode("promptfoo_failed_checks")
        .dropna(subset=["promptfoo_failed_checks"])
        .groupby("promptfoo_failed_checks", as_index=False)
        .size()
        .rename(columns={"promptfoo_failed_checks": "rubric", "size": "failed_trace_count"})
        .sort_values("failed_trace_count", ascending=False)
    )
    display(failed_metric_df)

    if not failed_metric_df.empty:
        fig = px.bar(
            failed_metric_df,
            x="failed_trace_count",
            y="rubric",
            orientation="h",
            title="Which lower-level rubric failed most often?",
            text="failed_trace_count",
            color="failed_trace_count",
            color_continuous_scale="Reds",
        )
        fig.update_layout(height=360, margin=dict(l=20, r=20, t=60, b=30))
        fig.update_yaxes(title="", categoryorder="total ascending")
        fig.update_xaxes(title="Failed traces")
        fig.show()


### 解读 Promptfoo 的输出

饼图是最简单的底层评分卡：它将通过所有评分标准检查的追踪与至少有一项检查失败的追踪区分开来。在实际的多智能体系统中，正是这一层告诉我们，在进行任何宏观分析之前，哪些运行值得关注。

失败评分标准柱状图回答了一个更有用的问题：哪些类型的智能体或工作流问题出现得最频繁？在该数据集中，最终决策质量是最主要的底层发现，策略正确性、审查恰当性和市场漂移意识也有出现。这表明宏观层应该少关注孤立的语法错误，而更多地关注重复出现的决策模式。

这就是通往宏观评估的桥梁。Promptfoo 为每条追踪提供局部评估标签。notebook 的其余部分则探讨这些标签在整个总体中如何组织起来。换句话说：智能体级评估产生原始信号，而宏观评估将大量此类信号转化为一张反复出现的系统行为地图。


## 4. 构建分析数据集

现在，我们将运行 bundle 规范化为两张分析表：

- `traces_df`：每次运行一行，包含元数据、结果、发现和文档字段。
- `events_df`：每个规范化追踪事件一行，包括交接、工具调用、状态事件、模型响应以及审查/发现标记。

我们还会构建追踪文档。文档是 BERTopic 风格部分将要聚类的建模对象。本 notebook 使用 `doc_structured_summary`，因为它既紧凑，又保留了场景、路由、状态转换、交接、发现和终止状态。

公开的分析路径是：

`case_type -> run_outcome -> eval_finding -> behavior_pattern`

前三个标签在聚类之前就已知，第四个标签在发现阶段之后才出现。


In [ ]:
OUTCOME_GROUP_MAP = {
    "completed": "successful_completion",
    "awaiting_review": "review_escalation",
    "blocked": "hard_failure",
    "failed": "hard_failure",
}
SEVERITY_BY_OUTCOME = {
    "successful_completion": ("low", 1.0),
    "review_escalation": ("medium", 2.0),
    "in_progress": ("medium", 1.5),
    "blocked": ("high", 2.5),
    "hard_failure": ("high", 3.0),
}


def local_bundle_path(result_row: dict[str, Any]) -> Path:
    return BUNDLE_DIR / Path(str(result_row["bundle_path"])).name


def load_normalized_bundle_tables(results: list[dict[str, Any]], limit: int | None = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    selected_rows = [row for row in results if result_run_id(row) and row.get("bundle_path")]
    if limit is not None:
        selected_rows = selected_rows[:limit]

    normalized = []
    for record_index, result_row in enumerate(selected_rows, start=1):
        bundle_path = local_bundle_path(result_row)
        if not bundle_path.is_file():
            continue
        bundle = read_json(bundle_path)
        normalized.append(normalize_bundle(bundle, result_row, record_index, bundle_path))

    trace_rows = [trace_row for trace_row, _ in normalized]
    event_rows = [event for _, trace_events in normalized for event in trace_events]
    traces = pd.DataFrame(trace_rows)
    events = pd.DataFrame(event_rows)

    if not events.empty:
        events["ts"] = pd.to_datetime(events["ts"], utc=True, errors="coerce")
        events["ended_at"] = pd.to_datetime(events["ended_at"], utc=True, errors="coerce")
        events = events.sort_values(["trace_id", "sequence_index", "ts", "event_id"]).reset_index(drop=True)
    return traces, events


load_started = perf_counter()
traces_df, events_df = load_normalized_bundle_tables(results_rows, limit=TRACE_LIMIT)
print(f"Loaded {len(traces_df):,} normalized traces and {len(events_df):,} normalized events in {perf_counter() - load_started:.1f}s.")

result_metadata_cols = [
    "run_id",
    "market_regime",
    "price_regime",
    "schedule_regime",
    "agent_version_set",
    "orchestrator_mode",
    "rogue_window_id",
    "factory_release_state",
    "trace_family",
    "specialist_activations",
    "environment_event_ids",
]
result_metadata_df = pd.DataFrame(results_rows)
result_metadata_cols = [column for column in result_metadata_cols if column in result_metadata_df.columns]
if "run_id" in result_metadata_cols:
    traces_df = traces_df.merge(
        result_metadata_df[result_metadata_cols].drop_duplicates(subset=["run_id"]),
        on="run_id",
        how="left",
        suffixes=("", "_result"),
    )
    for column in result_metadata_cols:
        if column == "run_id":
            continue
        result_col = f"{column}_result"
        if result_col in traces_df.columns:
            traces_df[column] = traces_df[column].combine_first(traces_df[result_col]) if column in traces_df.columns else traces_df[result_col]
            traces_df = traces_df.drop(columns=[result_col])

for column, default in {
    "market_regime": "unknown",
    "price_regime": "unknown",
    "schedule_regime": "unknown",
    "agent_version_set": "unknown",
    "orchestrator_mode": "unknown",
}.items():
    if column not in traces_df.columns:
        traces_df[column] = default

sqlite_enrichment_cols = [
    "run_id",
    "sqlite_status",
    "sqlite_terminal_state",
    "scenario_family",
    "validation_outcome",
    "review_status",
    "review_decision",
    "triage_outcome",
    "market_regime",
    "price_regime",
    "schedule_regime",
    "agent_version_set",
    "orchestrator_mode",
    "rogue_window_id",
    "factory_release_state",
    "trace_family",
    "loop_count",
    "retry_count",
    "arbitration_count",
    "compound_issue_count",
    "findings_count_sqlite",
]

if not sqlite_runs_df.empty:
    traces_df = traces_df.merge(sqlite_runs_df[sqlite_enrichment_cols], on="run_id", how="left", suffixes=("", "_sqlite"))

    for column in [
        "scenario_family",
        "validation_outcome",
        "review_status",
        "review_decision",
        "triage_outcome",
        "market_regime",
        "price_regime",
        "schedule_regime",
        "agent_version_set",
        "orchestrator_mode",
        "rogue_window_id",
        "factory_release_state",
        "trace_family",
        "loop_count",
        "retry_count",
        "arbitration_count",
        "compound_issue_count",
    ]:
        sqlite_col = f"{column}_sqlite"
        if sqlite_col in traces_df.columns:
            traces_df[column] = traces_df[sqlite_col].combine_first(traces_df.get(column))
            traces_df = traces_df.drop(columns=[sqlite_col])

    traces_df["runtime_status"] = traces_df["sqlite_status"].combine_first(traces_df["runtime_status"])
    traces_df["terminal_state"] = traces_df["sqlite_terminal_state"].combine_first(traces_df["terminal_state"])
    traces_df["findings_count"] = traces_df["findings_count_sqlite"].combine_first(traces_df["findings_count"]).fillna(0)
else:
    traces_df["findings_count"] = traces_df.get("findings_count", pd.Series(0, index=traces_df.index)).fillna(0)

traces_df["outcome_group"] = traces_df["runtime_status"].map(OUTCOME_GROUP_MAP).fillna("unknown")
traces_df["severity_label"] = traces_df["outcome_group"].map(lambda value: SEVERITY_BY_OUTCOME.get(value, ("medium", 2.0))[0])
traces_df["severity_weight"] = traces_df["outcome_group"].map(lambda value: SEVERITY_BY_OUTCOME.get(value, ("medium", 2.0))[1])
traces_df["has_failure"] = (
    traces_df["outcome_group"].ne("successful_completion")
    | traces_df["validation_outcome"].fillna("passed").ne("passed")
    | traces_df["findings_count"].fillna(0).gt(0)
)
traces_df["impact_score"] = (
    traces_df["severity_weight"].fillna(1.0)
    * (1.0 + traces_df["findings_count"].fillna(0))
    * (1.0 + traces_df["loop_count"].fillna(0) / 4.0)
)

documents_df = build_trace_documents(traces_df, events_df)
traces_with_docs_df = traces_df.merge(documents_df, on="trace_id", how="left")
labeled_traces_df = add_public_label_columns(traces_with_docs_df, promptfoo_labels_df=promptfoo_labels_df)
labeled_traces_df["eval_finding"] = labeled_traces_df["eval_finding"].apply(lambda value: clean_metric_name(value) or "none")
labeled_traces_df["promptfoo_failed"] = labeled_traces_df.get("promptfoo_pass").eq(False)

analysis_profile_df = pd.DataFrame(
    [
        ("normalized_traces", len(labeled_traces_df)),
        ("normalized_events", len(events_df)),
        ("case_types", labeled_traces_df["case_type"].nunique()),
        ("run_outcomes", labeled_traces_df["run_outcome"].nunique()),
        ("eval_findings", labeled_traces_df["eval_finding"].nunique()),
        ("promptfoo_failed_traces", int(labeled_traces_df["promptfoo_failed"].sum())),
        ("failure_or_review_traces", int(labeled_traces_df["has_failure"].sum())),
    ],
    columns=["metric", "value"],
)
display(analysis_profile_df)
display(labeled_traces_df[["run_id", "case_type", "run_outcome", "eval_finding", "market_regime", "agent_version_set", "impact_score"]].head(10))


### 解读分析概况

上面的概况确认了底层评估层已经关联到规范化后的追踪总体上。重要的数字包括：

- 规范化追踪数：我们可以检查的、有 bundle 支撑的总体；
- 规范化事件数：这些追踪背后的事件级证据；
- case type 数：生成器产生的场景覆盖范围；以及
- Promptfoo 失败或带有审查/失败信号的追踪数：与宏观发现最相关的底层信号总体。

确切的数量取决于你是运行完整的 notebook，还是为冒烟测试设置了 `MACRO_EVALS_TRACE_LIMIT`。示例行展示了 notebook 如何将原始数据简化为易读的标签。例如，一个以审查告终、带有 `final_decision_quality` 发现的 `pricing_exception_compound` 案例，现在就可以在 notebook 的其余部分中轻松追踪。


In [ ]:
def humanize_label(value: Any, max_len: int = 48) -> str:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        text = "missing"
    else:
        text = str(value)
    text = text.replace("_", " ")
    return text if len(text) <= max_len else text[: max_len - 3].rstrip() + "..."


def plot_label_sankey(frame: pd.DataFrame, columns: list[str], title: str, min_count: int = 1):
    working = frame[columns].copy()
    for column in columns:
        working[column] = working[column].fillna("missing").astype(str)

    node_labels: list[str] = []
    node_lookup: dict[tuple[str, str], int] = {}
    sources: list[int] = []
    targets: list[int] = []
    values: list[int] = []

    def node_id(column: str, value: str) -> int:
        key = (column, value)
        if key not in node_lookup:
            node_lookup[key] = len(node_labels)
            node_labels.append(f"{column}: {humanize_label(value)}")
        return node_lookup[key]

    for left, right in zip(columns[:-1], columns[1:]):
        pairs = working.groupby([left, right]).size().reset_index(name="count")
        pairs = pairs[pairs["count"].ge(min_count)]
        for _, row in pairs.iterrows():
            sources.append(node_id(left, row[left]))
            targets.append(node_id(right, row[right]))
            values.append(int(row["count"]))

    fig = go.Figure(
        data=[
            go.Sankey(
                arrangement="snap",
                node=dict(label=node_labels, pad=14, thickness=14),
                link=dict(source=sources, target=targets, value=values),
            )
        ]
    )
    fig.update_layout(title=title, height=620, margin=dict(l=20, r=20, t=60, b=20))
    return fig


flow_sample_df = labeled_traces_df[["case_type", "run_outcome", "eval_finding"]].copy()
plot_label_sankey(
    flow_sample_df,
    ["case_type", "run_outcome", "eval_finding"],
    "Before clustering: generated case -> run outcome -> lower-level finding",
    min_count=3,
).show()

label_crosswalk_df = (
    labeled_traces_df.groupby(["case_type", "run_outcome", "eval_finding"], dropna=False)
    .size()
    .reset_index(name="trace_count")
    .sort_values("trace_count", ascending=False)
    .head(18)
)
display(label_crosswalk_df)


### 第一张桑基图揭示了什么

第一张桑基图是聚类之前的视图。它展示了生成的 case type 如何流向运行结果和底层发现。

从左到右阅读：

- 从某个 `case_type` 出发的宽带表示该场景出现得很频繁；
- 分流到 `run_outcome` 的部分显示该场景倾向于完成、暂停、被阻断还是失败；
- 最终流入 `eval_finding` 的带状部分显示附带了哪些底层评分标准或运行时信号。

这对团队来说已经很有用了。业务读者可以借此判断仿真是否产生了恰当类型的压力。AI 工程师可以借此判断某些场景是否过多地产生了相同的底层发现。它还无法回答的是：这些发现是否代表同一种潜在的行为模式。这就是我们接下来要进行聚类的原因。


### 追踪文档：将运行转换为可比较的文本

原始的智能体追踪过于详细，无法直接聚类。它可能包含数百个事件、冗长的模型响应、工具负载以及重复的状态更新。文档构建步骤会将每次运行压缩为一个可比较的视图，同时保留对宏观评估重要的信息。

一份好的追踪文档包括：

- 业务设定（`case_type`、所选路由、活跃的环境信号）；
- 运行结果及严重程度；
- 重要的交接和专家激活；
- 审查/发现标记；
- 简短的状态转换摘要。

文档视图定义了聚类算法能够注意到哪些内容。包含智能体交接有助于宏观评估发现路由模式；包含环境信号有助于发现市场漂移类故障。因此，追踪文档的质量是评估设计的一部分，而不是一个机械的清理步骤。


### 故障与焦点事件术语表

原始追踪包含许多事件级标签。为了保持 notebook 的可读性，我们不要求读者学习所有标签。AgentTrace 风格的部分主要关注焦点事件：追踪中系统看起来需要关注的可见时刻。

在本仿真中，常见的焦点事件信号包括：

- `review finding`：审查或校验环节记录了一个问题。
- `review required` 或 `awaiting_review`：由于仿真的业务流程要求审查，运行被暂停。
- `failed` 或 `blocked`：运行到达了降级的终止状态。
- `triage route` 或改道信号：由于需要另一个负责方采取行动，工作流改变了方向。
- 工具警告或策略标记：结构化的工具输出指出了风险、歧义或策略约束。

这些是可观测性信号，而不是根本原因的证明。它们告诉诊断过程应该从哪里锚定并向后搜索。


In [ ]:
focus_event_guide_df = pd.DataFrame(
    [
        ("review finding", "An issue was recorded by review, validation, or a grading surface.", "Start from this when the trace has an explicit finding."),
        ("review required / awaiting_review", "The simulated business process paused for review.", "Check whether review was justified by the active risk."),
        ("failed / blocked", "The run ended in a degraded terminal state.", "Walk backward to the last handoff, tool, or specialist decision."),
        ("triage route / reroute", "The workflow changed ownership or path.", "Inspect whether routing matched the case type and environment signals."),
        ("tool warning / policy marker", "A structured tool exposed risk or policy context.", "Check whether later decisions used or ignored that signal."),
    ],
    columns=["focus_event_signal", "meaning", "how_to_use_it"],
)
display(focus_event_guide_df)


In [ ]:
example_candidates = labeled_traces_df[
    labeled_traces_df["promptfoo_failed"].fillna(False)
    & labeled_traces_df[DISCOVERY_DOC_COLUMN].fillna("").astype(str).str.len().gt(0)
]
if example_candidates.empty:
    example_candidates = labeled_traces_df[labeled_traces_df[DISCOVERY_DOC_COLUMN].fillna("").astype(str).str.len().gt(0)]

example_row = example_candidates.sort_values("impact_score", ascending=False).iloc[0]
display(Markdown(
    f"**Example trace document**  \n"
    f"`case_type={example_row['case_type']}` | `run_outcome={example_row['run_outcome']}` | "
    f"`eval_finding={example_row['eval_finding']}` | `impact_score={example_row['impact_score']:.2f}`"
))
print(str(example_row[DISCOVERY_DOC_COLUMN])[:2400])


上面的示例文档是将单条追踪渲染成的紧凑叙述。它有意比普通散文更密集，但比原始事件日志更容易比较。当你改造这一工作流时，请在文档构建上投入真正的时间。更好的文档通常比更复杂的聚类设置能产生更有用的行为模式。


## 5. BERTopic 风格的模式发现

发现过程受到 BERTopic 系列方法的启发。其高层思路是模块化的：

1. **将每份追踪文档表示为向量。** 如果追踪 $i$ 的文档是 $d_i$，嵌入模型会产生一个向量 $e_i = f(d_i)$。
2. **降低向量的几何维度。** 诸如 UMAP 之类的降维器将 $e_i$ 映射为一个低维点 $z_i$，并保留有用的局部邻域结构。
3. **对密集区域进行聚类。** 诸如 HDBSCAN 之类的密度聚类器将相邻的点分组，并可将离群点标记为噪声。
4. **表示每个主题。** 对于每个簇，计算能将该簇与语料库其余部分区分开来的词项。

本 notebook 使用辅助模块来保持实现紧凑，但主要的数学思想是清晰可见的：

- 当一条追踪的文档向量在降维空间中靠近其他追踪向量时，它就属于簇 $k$。
- 当一个词项在 $k$ 内频繁出现、而在其他地方较少出现时，它就有助于为簇 $k$ 打标签。
- 一个简单的类感知词项得分为：

$$
score(t, k) = tf(t, k) \times \log\left(\frac{1 + N}{1 + df(t)}\right)
$$

其中 $tf(t, k)$ 是词项 $t$ 在簇 $k$ 内的词频，$df(t)$ 是出现该词项的簇/文档数量，$N$ 是比较总体的规模。具体实现可能有所不同，但直觉是稳定的：标签应当描述是什么让一个簇与众不同。

最后，我们使用一个分诊指标对模式进行排序：

$$
impact\_score(k) = prevalence\_share(k) \times severity\_weighted\_prevalence(k)
$$

这并不是一个通用的风险公式，而是一个实用的优先级排序得分：一个模式既常见又严重时，就更加重要。


In [ ]:
discovery_input_df = labeled_traces_df.loc[
    labeled_traces_df["has_failure"]
    | labeled_traces_df["promptfoo_failed"].fillna(False)
    | labeled_traces_df["run_outcome"].isin(["review_needed", "blocked", "runtime_error"])
].copy()
discovery_input_df = discovery_input_df.loc[
    discovery_input_df[DISCOVERY_DOC_COLUMN].fillna("").astype(str).str.len().gt(0)
].copy()

if len(discovery_input_df) < 8:
    broader_input_df = labeled_traces_df.loc[
        labeled_traces_df[DISCOVERY_DOC_COLUMN].fillna("").astype(str).str.len().gt(0)
    ].copy()
    if len(broader_input_df) > len(discovery_input_df):
        display(Markdown(
            "The current sample has very few failure/review traces, so discovery is broadened to all traces with documents."
        ))
        discovery_input_df = broader_input_df

if len(discovery_input_df) < 2:
    raise ValueError("Macro discovery needs at least two trace documents. Increase MACRO_EVALS_TRACE_LIMIT or run the full notebook.")

effective_min_cluster_size = min(DISCOVERY_MIN_CLUSTER_SIZE, max(2, len(discovery_input_df) // 4))
effective_n_neighbors = min(30, max(2, len(discovery_input_df) - 1))

print(f"Discovery input traces: {len(discovery_input_df):,}")
print(f"Discovery min_cluster_size: {effective_min_cluster_size}")
print(f"Discovery n_neighbors: {effective_n_neighbors}")

discovery_started = perf_counter()
discovery = run_macro_discovery(
    discovery_input_df,
    document_column=DISCOVERY_DOC_COLUMN,
    min_cluster_size=effective_min_cluster_size,
    n_neighbors=effective_n_neighbors,
    top_n_terms=8,
    random_state=RANDOM_STATE,
    failure_only=False,
)
discovery_seconds = perf_counter() - discovery_started

topic_info_df = discovery.topic_info_df.copy()
work_df = discovery.trace_topic_df.copy()
work_df["behavior_pattern"] = work_df["topic_label"].fillna(work_df["topic_id"].astype(str))
topic_info_df["behavior_pattern"] = topic_info_df["topic_label"].fillna(topic_info_df["topic_id"].astype(str))

display(
    pd.DataFrame(
        [
            ("discovery_seconds", round(discovery_seconds, 2)),
            ("input_traces", len(discovery_input_df)),
            ("topics_including_noise", topic_info_df["topic_id"].nunique()),
            ("non_noise_patterns", int(topic_info_df["topic_id"].ne(-1).sum())),
        ],
        columns=["metric", "value"],
    )
)

display(
    topic_info_df[
        ["topic_id", "behavior_pattern", "trace_count", "prevalence", "impact_score", "dominant_owner", "keywords_text"]
    ].sort_values("impact_score", ascending=False).head(12)
)


### 解读发现结果

发现摘要告诉我们有多少条追踪被聚类，以及恢复出了多少个非噪声的行为模式。我们只对已经带有失败、审查、运行时或 Promptfoo 信号的追踪运行发现过程，因为本 cookbook 关注的是系统中需要关注的地方。

主题表应当作为一块分诊看板来阅读：

- `trace_count` 和 `prevalence` 告诉我们该模式出现的频率。
- `severity_weighted_prevalence` 告诉我们该模式中的追踪往往有多严重。
- `impact_score` 将普遍性和严重性结合成一个排序。
- `dominant_owner` 是一个启发式的负责方标签，而不是正式的分派。
- `keywords_text` 给出了让该模式与众不同的词项。

一个高影响的行为模式并不自动意味着缺陷。它是审查者应当首先查看的地方，因为该模式频繁出现、后果严重，或两者兼而有之。


In [ ]:
impact_explanation_df = (
    topic_info_df[topic_info_df["topic_id"].ne(-1)]
    [["behavior_pattern", "trace_count", "prevalence", "severity_weighted_prevalence", "impact_score"]]
    .sort_values("impact_score", ascending=False)
    .head(8)
    .copy()
)
impact_explanation_df["formula"] = "prevalence x severity_weighted_prevalence"
display(impact_explanation_df)


上表将影响得分具体化了。一个模式可能因为出现在许多追踪中、因为集中了较高严重程度的追踪，或者两者兼有而排名靠前。在汽车配置器场景中，这有助于将罕见的边缘案例与可能影响大量订单的反复出现的运营行为区分开来。


In [ ]:
leaderboard_fig = plot_topic_leaderboard(topic_info_df[topic_info_df["topic_id"].ne(-1)].copy(), top_n=10)
leaderboard_fig.update_layout(title_text="Behavior patterns by weighted impact")
leaderboard_fig.show()

scatter_df = discovery.topic_assignments.copy()
if "topic_label" in scatter_df.columns:
    scatter_df["behavior_pattern"] = scatter_df["topic_label"].fillna(scatter_df["topic_id"].astype(str))
    scatter_fig = plot_topic_scatter(
        scatter_df,
        color_col="behavior_pattern",
        hover_cols=("run_id", "case_type", "run_outcome", "eval_finding"),
        title="Trace map after discovery",
    )
    scatter_fig.update_layout(legend_title_text="Behavior pattern")
    scatter_fig.show()


### 解读排行榜与追踪地图

排行榜是一个组合视图：它按加权影响对行为模式进行排序。用它来决定哪个模式最值得人工首先关注。

追踪地图是一个几何视图：每个点代表一份追踪文档，并被放置在文本相似的追踪附近。相邻的点通常共享路由路径、发现或环境信号。颜色表示发现的行为模式。请将这张地图视为诊断工具，而不是精确的地理图。它的作用是揭示在表格中可能难以看出的簇和离群点。

在该数据集中，诸如履约改道、定价漂移、合规关卡以及轮毂/内饰不匹配等模式，都对应着可识别的业务问题。这是底层评估首次转变为宏观层面叙事的时刻：重复的智能体行为在许多案例中变得可见。


In [ ]:
case_pattern_df = (
    work_df[work_df["topic_id"].ne(-1)]
    .groupby(["case_type", "behavior_pattern"], dropna=False)
    .size()
    .reset_index(name="trace_count")
)
if not case_pattern_df.empty:
    case_totals = case_pattern_df.groupby("case_type")["trace_count"].transform("sum")
    case_pattern_df["share_within_case_type"] = case_pattern_df["trace_count"] / case_totals
    display(case_pattern_df.sort_values(["share_within_case_type", "trace_count"], ascending=[False, False]).head(20))

    heatmap_input_df = case_pattern_df.rename(
        columns={
            "case_type": "slice_value",
            "share_within_case_type": "slice_share",
        }
    )
    heatmap_input_df["lift"] = heatmap_input_df["slice_share"] / heatmap_input_df.groupby("behavior_pattern")["trace_count"].transform(lambda s: s.sum() / len(work_df))
    plot_topic_heatmap(
        heatmap_input_df,
        row_col="behavior_pattern",
        col_col="slice_value",
        value_col="slice_share",
        title="Behavior pattern concentration by generated case type",
        top_n_rows=8,
        top_n_cols=10,
    ).show()

pattern_eval_df = (
    work_df[work_df["topic_id"].ne(-1)]
    .assign(promptfoo_failed=lambda df: df["promptfoo_pass"].eq(False))
    .groupby(["behavior_pattern", "eval_finding"], dropna=False)
    .agg(trace_count=("trace_id", "count"), promptfoo_fail_rate=("promptfoo_failed", "mean"))
    .reset_index()
    .sort_values(["trace_count", "promptfoo_fail_rate"], ascending=False)
    .head(20)
)
display(pattern_eval_df)


### 解读 Case-Type 热力图

热力图要回答的是：哪些生成的场景集中了哪些行为模式？

将每一行视为一个行为模式，每一列视为一个 case type。颜色越深或数值越大，表示该模式在该场景切片中越常见。这有助于区分预期行为和意外行为。例如，履约改道模式在供应商替换或产能类案例中可能是意料之中的，但在正常案例中就更值得怀疑。

图表下方的表格将模式与底层发现关联起来。如果某个行为模式反复带有 `final_decision_quality` 发现，AI 工程师可能会检查提示词、工具 schema 或交接策略。如果该模式对应于某个特定业务的 case type，产品或运营利益相关者就可以追问仿真的策略本身是否切合实际。


### 跨切片比较模式

这一步出现在这里，是因为 BERTopic 风格的发现过程刚刚为每条有风险的追踪分配了一个 `behavior_pattern`。在聚类之前，我们可以比较生成的案例、结果和底层评估发现。聚类之后，我们就可以提出一个更有用的宏观评估问题：**每个被发现的行为模式集中在哪里？**

这种比较并不是 BERTopic 论文中的核心公式，而是我们在主题分配之后应用的一个简单的群组分析层。其思路是比较两个占比：

- **总体模式占比**：在所有被聚类的追踪中，属于该行为模式的占多少？
- **切片模式占比**：在某个切片内（例如 `case_type = supplier_substitution_compound`），属于该行为模式的占多少？

然后我们计算：

$$
lift = \frac{slice\ pattern\ share}{overall\ pattern\ share}
$$

提升度（lift）为 `1.0` 表示该模式在该切片中出现的频率与总体大致相同。提升度高于 `1.0` 表示该模式集中在该切片中。提升度低于 `1.0` 表示它在那里较少出现。

在宏观评估中，这是从发现走向行动的桥梁。当我们能够说出一个行为模式出现在哪里时——某个生成的场景、某个智能体版本、某种编排模式、某种市场状态或某个审查状态——它就更容易被调查。


In [ ]:
slice_lift_source_df = work_df[work_df["topic_id"].ne(-1)].copy()
overall_pattern_share = (
    slice_lift_source_df["behavior_pattern"]
    .value_counts(normalize=True)
    .rename("overall_pattern_share")
    .reset_index()
    .rename(columns={"index": "behavior_pattern"})
)
slice_pattern_counts_df = (
    slice_lift_source_df.groupby(["case_type", "behavior_pattern"], dropna=False)
    .size()
    .reset_index(name="trace_count")
)
slice_totals = (
    slice_lift_source_df.groupby("case_type", dropna=False)
    .size()
    .rename("slice_total")
    .reset_index()
)
slice_lift_df = (
    slice_pattern_counts_df
    .merge(slice_totals, on="case_type", how="left")
    .merge(overall_pattern_share, on="behavior_pattern", how="left")
)
slice_lift_df["slice_pattern_share"] = slice_lift_df["trace_count"] / slice_lift_df["slice_total"]
slice_lift_df["lift"] = slice_lift_df["slice_pattern_share"] / slice_lift_df["overall_pattern_share"].replace(0, np.nan)
slice_lift_view_df = (
    slice_lift_df[slice_lift_df["trace_count"].ge(5)]
    .sort_values(["lift", "trace_count"], ascending=[False, False])
    .head(12)
    .loc[:, ["case_type", "behavior_pattern", "trace_count", "slice_pattern_share", "overall_pattern_share", "lift"]]
)
display(slice_lift_view_df)


上表应当被视为一个调查队列。它突出显示了在某个给定 `case_type` 中异常集中的行为模式，同时要求至少有少量支撑追踪。例如，如果某个路由模式在供应商替换案例中比在总体中常见得多，这就表明团队应当先检查供应商工具、采购交接和履约策略，而不是将该模式视为一个笼统的系统问题。


In [ ]:
plot_label_sankey(
    work_df[["case_type", "run_outcome", "eval_finding", "behavior_pattern"]].copy(),
    ["case_type", "run_outcome", "eval_finding", "behavior_pattern"],
    "After clustering: generated case -> outcome -> eval finding -> behavior pattern",
    min_count=4,
).show()

public_label_view_df = (
    work_df[["run_id", "case_type", "run_outcome", "eval_finding", "behavior_pattern", "impact_score"]]
    .sort_values("impact_score", ascending=False)
    .head(12)
)
display(public_label_view_df)


### 第二张桑基图补充了什么

第二张桑基图将发现的 `behavior_pattern` 作为最后一步加入：

`case_type -> run_outcome -> eval_finding -> behavior_pattern`

这是宏观评估的关键一步。前三个标签描述了生成的设定、结局和局部症状。最后一个标签显示这些局部症状是否会收敛为少数几个重复出现的运作模式。

业务利益相关者可以借此提问：“哪些订单场景造成了最多的重复性运营问题？”AI 工程师可以借此提问：“哪些底层发现实际上是同一种路由或决策模式？”两种视角都很有用，而桑基图为它们提供了一张共享的地图。


## 6. AgentTrace 风格的诊断

发现过程告诉我们什么在重复出现，诊断过程则要回答应该首先检查哪里。

对于选定的行为模式，我们重建一个轻量级的执行图：

$$
G = (V, E)
$$

其中每个节点 $v \in V$ 是一个规范化的追踪事件，每条边 $e \in E$ 通过时间顺序、交接、工具调用和邻近的执行上下文将事件连接起来。然后我们选择一个焦点事件，也称为锚点。在本仿真中，焦点事件通常是审查/发现标记、与失败相关的状态或后期决策事件。

诊断过程从该锚点出发，在图中向后遍历，并对上游嫌疑项打分。该得分被有意设计为可解释的：

$$
suspect\_score =
0.4 \cdot proximity +
0.3 \cdot frequency +
0.2 \cdot bridge +
0.1 \cdot role
$$

- **Proximity（接近度）**奖励靠近焦点事件的事件。
- **Frequency（频率）**奖励在同一行为模式的抽样追踪中反复出现的事件。
- **Bridge（桥接度）**奖励连接执行图不同部分的事件。
- **Role（角色）**奖励其智能体/工具角色与该发现可能相关的事件。

这并不是因果关系的证明，而是一种方法，将“这个模式很重要”转化为“首先检查这些智能体、工具、交接或审查策略”。


In [ ]:
focus_topic = pick_focus_topic(topic_info_df, exclude_noise=True)
focus_topic_id = focus_topic["topic_id"]

display(Markdown(
    f"Investigating behavior pattern `{focus_topic['behavior_pattern']}` "
    f"from topic `{focus_topic_id}` with `{int(focus_topic['trace_count'])}` traces."
))
display(focus_topic[["topic_id", "behavior_pattern", "trace_count", "prevalence", "impact_score", "dominant_owner", "keywords_text"]].to_frame("value"))

root_cause = drill_down_topic_root_causes(
    discovery,
    events_df=events_df,
    topic_id=focus_topic_id,
    top_n_traces=12,
    max_depth=5,
)

def public_suspect_label(value: Any) -> str:
    text = str(value or "unknown")
    if text.startswith("failure: "):
        return "eval/review signal: " + text.removeprefix("failure: ")
    if text.startswith("handoff: "):
        return "handoff involving " + text.removeprefix("handoff: ")
    if text.startswith("function: "):
        return "tool/function call by " + text.removeprefix("function: ")
    if text.startswith("response: "):
        return "agent response by " + text.removeprefix("response: ")
    return text

if not root_cause.suspect_summary.empty:
    suspect_display_df = root_cause.suspect_summary.head(12).copy()
    suspect_display_df["reader_label"] = suspect_display_df["suspect_label"].apply(public_suspect_label)
    suspect_display_df["is_eval_or_review_signal"] = suspect_display_df["reader_label"].str.startswith("eval/review signal")
    display_columns = [
        column
        for column in [
            "reader_label",
            "node_kind",
            "agent_name",
            "tool_name",
            "lane_label",
            "mean_score",
            "trace_coverage_share",
        ]
        if column in suspect_display_df.columns
    ]
    display(suspect_display_df[display_columns])

    focus_signal_df = suspect_display_df[suspect_display_df["is_eval_or_review_signal"]].head(1)
    operational_suspect_df = suspect_display_df[~suspect_display_df["is_eval_or_review_signal"]].head(1)
    if not focus_signal_df.empty:
        focus_signal = focus_signal_df.iloc[0]
        operational_label = (
            operational_suspect_df.iloc[0]["reader_label"]
            if not operational_suspect_df.empty
            else "the next highest-ranked non-review event"
        )
        display(Markdown(
            "#### Reading the focus signal\n\n"
            f"The leading signal is `{focus_signal['reader_label']}` from `{focus_signal.get('agent_name', 'unknown')}`. "
            "In this simulation, a review finding means that a specialist or review surface recorded a structured issue while processing one customer order. "
            "For a fulfillment-reroute pattern, that signal is best read as the point where the workflow says: this order has enough supply, routing, policy, or review risk to deserve attention. "
            f"The first operational inspection target after that signal is `{operational_label}`."
        ))

    suspect_plot_df = root_cause.suspect_summary.head(10).copy()
    suspect_plot_df["suspect_label"] = suspect_plot_df["suspect_label"].apply(public_suspect_label)
    plot_suspect_leaderboard(suspect_plot_df).show()
else:
    display(Markdown("No repeated upstream suspects were recovered for this behavior pattern."))


### 解读嫌疑项排行榜

焦点行为模式是按影响得分选出的。根据你运行的是完整数据集还是较小的冒烟测试样本，选出的模式可能不同，但阅读过程是相同的：从影响最大的模式开始，然后检查哪些审查信号、交接、工具或专家响应反复出现在焦点事件附近。

像 `eval/review signal: review finding` 这样的行并不神秘。在仿真中，审查发现是一个结构化标记，当某个专家或审查环节观察到应当影响订单决策的问题时就会产生。它是我们向后追溯的终点：即工作流已经积累了足够证据，可以说“这个订单需要关注”的那一刻。

更具可操作性的行是该标记周围的运营事件：涉及编排器的交接、监控或编排智能体的工具/函数调用、采购计划交接，以及相关的专家响应。在宏观评估指向该模式之后，这些就是人工应当检查的地方。

从技术角度来看，这一输出告诉 AI 工程师应该检查哪里：

- 相关智能体的指令和工具契约；
- 围绕重复转换的交接规则；
- 系统记录审查标记是否过早或过晚；
- 某个工具输出是被忽略了还是被赋予了过高的权重。

从业务角度来看，同样的输出告诉运营或产品利益相关者哪个业务职能似乎是该模式的负责方。履约、定价、合规或澄清类的模式，应当让相应的业务负责人参与下一轮审查，而不仅仅是提示词工程师。


In [ ]:
if root_cause.selected_path_nodes:
    plot_root_cause_story(root_cause.__dict__, title="Representative path into the focus event").show()

if not root_cause.representative_trace_window.empty:
    plot_trace_swimlane(root_cause.representative_trace_window, title="Representative focus-event window").show()

if not root_cause.suspect_summary.empty:
    summary_suspects_df = root_cause.suspect_summary.copy()
    summary_suspects_df["reader_label"] = summary_suspects_df["suspect_label"].apply(public_suspect_label)
    review_signal_df = summary_suspects_df[summary_suspects_df["reader_label"].str.startswith("eval/review signal")].head(1)
    operational_target_df = summary_suspects_df[~summary_suspects_df["reader_label"].str.startswith("eval/review signal")].head(1)
    review_signal = review_signal_df.iloc[0] if not review_signal_df.empty else summary_suspects_df.iloc[0]
    operational_target = operational_target_df.iloc[0] if not operational_target_df.empty else summary_suspects_df.iloc[0]
    display(Markdown(
        "### Diagnosis summary\n\n"
        f"For behavior pattern `{focus_topic['behavior_pattern']}`, the focus signal is "
        f"`{review_signal['reader_label']}`. In the auto-order simulation, this means the trace reached a review/eval checkpoint where a specialist found an issue that could affect fulfillment or release. "
        f"The first operational target to inspect is `{operational_target['reader_label']}`. "
        "Read the story strip and swimlane as the path into that checkpoint: which agents handled the order, which handoffs occurred before the marker, and whether the workflow used the right supply, routing, and review signals before deciding what to do next."
    ))


### 解读故事条与泳道图

故事条是一条通往焦点事件的路径。在这次运行中，焦点事件是所选行为模式内部的一个审查/评估检查点。它代表仿真的业务流程在表明：这个订单存在值得审查的问题。

泳道视图保留了更多的时间结构。它按泳道或智能体展示焦点事件周围的事件窗口，并突出显示焦点事件。随着订单在智能体群中流转，从左到右阅读：

- 在审查发现之前，是哪个专家处理了该订单？
- 编排器是否在正确的时间路由经过了正确的业务负责方？
- 是否有某个工具/函数调用提供了本应改变订单决策的信息？
- 审查是否发生在工作流确定放行、改道、定价、合规或客户沟通建议之前？

对于业务读者，这张图将一个抽象的模式转化为一个运营故事：这一组订单反复到达类似的审查点。对于 AI 工程师，它缩小了下一步调试的范围：检查审查标记周围的编排和交接路径，尤其是诊断摘要中突出显示的第一个非审查类嫌疑项。


## 7. 我们学到了什么以及下一步该做什么

本 cookbook 经历了四个层级的证据：

1. **仿真设定：** 业务在不断变化的供应、定价、产能、合规和市场条件下生成了电动汽车订单案例。
2. **底层评估：** Promptfoo 提供了智能体/工作流级的评估信号：决策质量、策略正确性、路由、市场意识和审查恰当性。
3. **宏观发现：** BERTopic 风格的聚类将底层发现归组为反复出现的行为模式，并按影响进行排序。
4. **追踪诊断：** AgentTrace 风格的图分析检查了一个高影响模式，并识别出反复出现的上游嫌疑项。

这种方法通过将人工注意力引导到既频繁又后果严重的模式上来实现规模化。审查者无需从头到尾阅读数百条追踪，而是可以从一个行为模式出发，检查有代表性的示例，并决定哪个智能体、工具、交接或业务规则值得跟进。

AI 工程团队的实用后续步骤：

- 将最清晰的底层评估失败案例提升为回归测试套件；
- 审查少量自动评分样本，以校准评分标准的严格程度；
- 按模型版本、提示词版本和编排模式跟踪行为模式；
- 为影响最大的模式指派业务负责人；
- 在修改系统之前，检查排名靠前的嫌疑智能体、工具和交接。

业务利益相关者的实用后续步骤：

- 判断生成的 case type 是否与真实的运营风险相匹配；
- 检查高影响模式是否对应重要的客户或运营结果；
- 验证审查阈值是否产生了预期的业务行为；
- 使用桑基图和热力图视图，确定哪些场景需要更好的策略或流程设计。

核心结论很简单：智能体级评估告诉我们哪些局部行为看起来有风险，而宏观评估告诉我们这些风险在系统规模上会演变成什么。

**延伸阅读**

- OpenAI Agents SDK 文档：[Agents SDK 用例](https://developers.openai.com/api/docs/libraries#use-the-agents-sdk)
- BERTopic 文档：[概览](https://maartengr.github.io/BERTopic/index.html)和[算法详解](https://maartengr.github.io/BERTopic/algorithm/algorithm.html)
- Promptfoo 文档：[OpenAI Agents provider](https://www.promptfoo.dev/docs/providers/openai-agents/) 和[聊天对话](https://www.promptfoo.dev/docs/configuration/chat/)
- AgentTrace 论文：[Causal Graph Tracing for Root Cause Analysis in Deployed Multi-Agent Systems](https://arxiv.org/abs/2603.14688)


## 贡献者
本 cookbook 是 OpenAI 与 Slalom 共同协作的成果。

- [Shikhar Kwatra](https://www.linkedin.com/in/shikharkwatra/)
- [Will Thieme](https://www.linkedin.com/in/will-thieme-8689758b/)
- [Bradley Strauss](https://www.linkedin.com/in/bradleyjstrauss/)